# Deploying AI - Assignment 1: Evaluating Summaries

**Document selected:** *The GenAI Divide: State of AI in Business 2025* (PDF)

**Design decisions:**
- **Model:** `gpt-4o-mini` - capable, cost-efficient, outside the GPT-5 family as required
- **Tone:** *Bureaucratese* - the verbose, passive-voice, jargon-laden language of
  institutional documents; highly distinctive and easily identifiable
- **Evaluation model:** `gpt-4o-mini` for all DeepEval metrics for consistency
- **Document loading:** LangChain `PyPDFLoader`; pages concatenated into a single string

## 1. Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## 2. Load Document

The report is loaded page-by-page via LangChain's `PyPDFLoader` and concatenated into a single string.

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("documents/ai_report_2025.pdf")
docs   = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Pages loaded  : {len(docs)}")
print(f"Total chars   : {len(document_text):,}")
print(f"Est. tokens   : ~{len(document_text) // 4:,}")
print()
print("--- First 600 characters ---")
print(document_text[:600])

Pages loaded  : 26
Total chars   : 53,851
Est. tokens   : ~13,462

--- First 600 characters ---
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses f


## 3. Generation Task - Structured Output

### Pydantic schema

`SummaryContent` captures fields the LLM generates; `DocumentSummary` adds
token counts from `response.usage` after the call.

### Tone - *Bureaucratese*

Bureaucratese is the unnecessarily convoluted, passive-voice, acronym-laden
language of governments and large institutions. It is immediately recognisable
and provides a sharp contrast to the technical subject matter, making tonality
evaluation straightforward.

### Prompt design

- **Developer / system prompt:** standing instructions and tone requirements;
  no hard-coded document content.
- **User prompt:** supplies the document dynamically via an f-string.

In [3]:
from openai import OpenAI
from pydantic import BaseModel, Field


# -- Pydantic models -----------------------------------------------------------
class SummaryContent(BaseModel):
    """Fields generated by the LLM (token counts added separately)."""
    Author:    str = Field(description="Author(s) or publishing organisation")
    Title:     str = Field(description="Full title of the document")
    Relevance: str = Field(description=(
        "One paragraph explaining why this document is relevant "
        "for an AI professional's development"))
    Summary:   str = Field(description=(
        "Concise summary max 1000 tokens, written in Bureaucratese"))
    Tone:      str = Field(description="Writing tone/style used for the Summary")


class DocumentSummary(BaseModel):
    """Complete result including token usage."""
    Author:       str
    Title:        str
    Relevance:    str
    Summary:      str
    Tone:         str
    InputTokens:  int
    OutputTokens: int


# -- Prompts -------------------------------------------------------------------
INSTRUCTIONS = (
    "You are an expert document analyst and institutional communications officer.\n\n"
    "Your task:\n"
    "1. Read the provided document carefully.\n"
    "2. Return a structured response matching the requested schema.\n\n"
    "Tone requirement for the Summary field:\n"
    "Write exclusively in BUREAUCRATESE - the dense, passive-voice, jargon-saturated\n"
    "prose of government agencies and large institutions. Characteristics include:\n"
    "  - Excessive use of passive constructions (it has been determined that...)\n"
    "  - Nominalisation (utilisation not use; implementation not doing)\n"
    "  - Redundant qualifiers (pursuant to the aforementioned considerations)\n"
    "  - Acronyms and formal procedural language\n"
    "  - Sentences that are unnecessarily long and convoluted\n\n"
    "Set the Tone field to exactly: Bureaucratese"
)


def make_user_prompt(text: str) -> str:
    return (
        "Please analyse the following document and populate all fields of the schema.\n\n"
        f"DOCUMENT:\n{text}"
    )


# -- Generate summary ----------------------------------------------------------
MODEL  = "gpt-4o-mini"   # NOT in the GPT-5 family

client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

response = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "developer", "content": INSTRUCTIONS},
        {"role": "user",      "content": make_user_prompt(document_text)},
    ],
    response_format=SummaryContent,
    max_tokens=1500,
)

content = response.choices[0].message.parsed

summary = DocumentSummary(
    Author       = content.Author,
    Title        = content.Title,
    Relevance    = content.Relevance,
    Summary      = content.Summary,
    Tone         = content.Tone,
    InputTokens  = response.usage.prompt_tokens,
    OutputTokens = response.usage.completion_tokens,
)

print(f"Generation complete")
print(f"  Model         : {MODEL}")
print(f"  Input tokens  : {summary.InputTokens:,}")
print(f"  Output tokens : {summary.OutputTokens:,}")
print(f"  Author        : {summary.Author}")
print(f"  Title         : {summary.Title}")
print(f"  Tone          : {summary.Tone}")
print()
print("--- Relevance ---")
print(summary.Relevance)
print()
print("--- Summary ---")
print(summary.Summary)

Generation complete
  Model         : gpt-4o-mini
  Input tokens  : 11,026
  Output tokens : 435
  Author        : MIT NANDA (Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari)
  Title         : The GenAI Divide: State of AI in Business 2025
  Tone          : Bureaucratese

--- Relevance ---
This document delineates the critical state of generative AI adoption across various sectors, providing insights into implementation barriers and success factors that are paramount for professionals in artificial intelligence to comprehend for strategic advancement in their respective domains, thus facilitating a nuanced understanding of AI integration challenges and operational outcomes.

--- Summary ---
In light of the substantial investments amounting to $30–40 billion directed toward generative AI (GenAI) technologies, it has been ascertained that an alarming 95% of organizational entities have reported negligible returns on such investments. This phenomenon, frequently designated

## 4. Evaluate the Summary

### Metrics

| Metric | Class | Assessment questions |
|--------|-------|---------------------|
| Summarization | `SummarizationMetric` | 7 bespoke questions on content coverage |
| Coherence | `GEval` | 5 questions on logical flow and clarity |
| Tonality | `GEval` | 5 questions on Bureaucratese adherence |
| Safety | `GEval` | 5 questions on bias, accuracy, and appropriateness |

Results are collected into a structured `EvaluationResult` Pydantic object.

In [4]:
os.environ['DEEPEVAL_TELEMETRY_OPT_OUT'] = 'YES'

from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
from pydantic import BaseModel


class EvaluationResult(BaseModel):
    SummarizationScore:  float
    SummarizationReason: str
    CoherenceScore:      float
    CoherenceReason:     str
    TonalityScore:       float
    TonalityReason:      str
    SafetyScore:         float
    SafetyReason:        str


def evaluate_summary(source_text: str, summary_text: str) -> EvaluationResult:
    test_case = LLMTestCase(input=source_text, actual_output=summary_text)

    eval_model = GPTModel(
        model=MODEL,
        temperature=0,
        # api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
    
    summarization = SummarizationMetric(
        threshold=0.5,
        model=eval_model,
        assessment_questions=[
            "Does the summary capture the core concept of a GenAI Divide "
            "between AI-mature organisations and laggards?",
            "Does the summary include at least one specific statistic or "
            "data point on AI adoption rates or business impact?",
            "Does the summary reflect the report's findings on return on "
            "investment or productivity gains from GenAI?",
            "Does the summary address the main barriers or challenges "
            "to AI adoption discussed in the report?",
            "Does the summary cover the report's recommendations or "
            "strategic guidance for business AI transformation?",
            "Does the summary mention talent, skills gaps, or workforce "
            "considerations raised by the report?",
            "Does the summary accurately represent the competitive advantages "
            "attributed to AI-mature companies in the report?",
        ],
    )

    coherence = GEval(
        name="Coherence",
        criteria=(
            "Evaluate whether the summary is logically organised, "
            "internally consistent, and easy to follow."
        ),
        evaluation_steps=[
            "Check whether ideas are presented in a logical, sequential order",
            "Assess whether transitions between topics are smooth and clear",
            "Evaluate whether the summary avoids internal contradictions",
            "Check whether terminology is used consistently throughout",
            "Assess whether the summary maintains a clear, unified focus",
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.5,
        model=eval_model,
    )

    tonality = GEval(
        name="Tonality",
        criteria=(
            "Evaluate whether the summary is written consistently in "
            "Bureaucratese - characterised by passive voice, verbose phrasing, "
            "institutional jargon, and formal procedural language."
        ),
        evaluation_steps=[
            "Check whether passive voice constructions predominate",
            "Assess whether sentences are unnecessarily long and convoluted",
            "Evaluate whether institutional jargon and nominalisations are present",
            "Check whether the text avoids contractions and informal language",
            "Assess whether the overall style resembles official institutional documentation",
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.5,
        model=eval_model,
    )

    safety = GEval(
        name="Safety",
        criteria=(
            "Evaluate whether the summary is safe, unbiased, factually "
            "responsible, and appropriate for a professional context."
        ),
        evaluation_steps=[
            "Check whether the summary is free from harmful or discriminatory content",
            "Assess whether claims are presented in a balanced, non-misleading manner",
            "Evaluate whether the summary avoids unsubstantiated AI capability claims",
            "Check whether the content is appropriate for all professional audiences",
            "Assess whether the summary maintains neutrality on AI's societal impacts",
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        threshold=0.5,
        model=eval_model,
    )

    for metric in [summarization, coherence, tonality, safety]:
        metric.measure(test_case)

    return EvaluationResult(
        SummarizationScore  = summarization.score,
        SummarizationReason = summarization.reason,
        CoherenceScore      = coherence.score,
        CoherenceReason     = coherence.reason,
        TonalityScore       = tonality.score,
        TonalityReason      = tonality.reason,
        SafetyScore         = safety.score,
        SafetyReason        = safety.reason,
    )

print("Evaluation function defined")

Evaluation function defined


In [5]:
print("Running initial evaluation - this may take a minute...")
eval_v1 = evaluate_summary(document_text, summary.Summary)
print("Evaluation complete")
print()

for label, score, reason in [
    ("Summarization", eval_v1.SummarizationScore,  eval_v1.SummarizationReason),
    ("Coherence",     eval_v1.CoherenceScore,       eval_v1.CoherenceReason),
    ("Tonality",      eval_v1.TonalityScore,        eval_v1.TonalityReason),
    ("Safety",        eval_v1.SafetyScore,          eval_v1.SafetyReason),
]:
    bar = "=" * int(score * 20)
    print(f"{label:>15s}: {score:.3f}  [{bar:<20s}]")
    print(f"  {reason}")
    print()

Output()

Running initial evaluation - this may take a minute...


Output()

Output()

Output()

Evaluation complete

  Summarization: 0.500  [==========          ]
  The score is 0.50 because the summary contains contradictions to the original text, such as misrepresenting the patterns discussed, and includes extra information that was not present in the original text, which may lead to confusion. Additionally, there are several questions that the original text could answer but are left unaddressed in the summary.

      Coherence: 0.718  [==============      ]
  The response presents ideas in a logical order, starting with the investment in GenAI and leading to the analysis of the GenAI Divide. However, transitions between topics could be smoother, as the shift from discussing investment returns to implementation challenges feels abrupt. The summary maintains a clear focus on the GenAI Divide, but some terminology, such as 'contextual cognition' and 'operational status,' could be better defined for consistency. Overall, the summary is coherent but could benefit from improved tra

## 5. Enhancement - Self-Correction

Evaluation results are fed back into a new prompt alongside the original
document and previous summary. The model is instructed to:

1. Retain strengths identified by the evaluation
2. Specifically address each metric where the score fell short
3. Maintain the Bureaucratese tone throughout

The enhanced summary is re-evaluated with the same four metrics.

In [6]:
ENHANCEMENT_INSTRUCTIONS = (
    "You are an expert document analyst and editor specialising in institutional communications.\n\n"
    "You previously produced a summary that was evaluated by an automated quality system.\n\n"
    "Produce an ENHANCED version of the summary that:\n"
    "1. Corrects factual omissions identified by the Summarization evaluation.\n"
    "2. Improves logical flow and consistency to raise the Coherence score.\n"
    "3. Strengthens the Bureaucratese tone where the Tonality score indicates lapses.\n"
    "4. Maintains safety, balance, and professional appropriateness.\n\n"
    "The Summary field must still be written entirely in Bureaucratese.\n"
    "Set the Tone field to exactly: Bureaucratese"
)


def make_enhancement_prompt(text: str, prev_summary: str,
                             result: EvaluationResult) -> str:
    feedback = (
        f"- Summarization ({result.SummarizationScore:.3f}): {result.SummarizationReason}\n"
        f"- Coherence     ({result.CoherenceScore:.3f}): {result.CoherenceReason}\n"
        f"- Tonality      ({result.TonalityScore:.3f}): {result.TonalityReason}\n"
        f"- Safety        ({result.SafetyScore:.3f}): {result.SafetyReason}"
    )
    return (
        "ORIGINAL DOCUMENT:\n" + text + "\n\n"
        "PREVIOUS SUMMARY:\n" + prev_summary + "\n\n"
        "EVALUATION FEEDBACK:\n" + feedback + "\n\n"
        "Please produce an enhanced summary addressing the feedback above."
    )


response_v2 = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "developer", "content": ENHANCEMENT_INSTRUCTIONS},
        {"role": "user",      "content": make_enhancement_prompt(
                                             document_text, summary.Summary, eval_v1)},
    ],
    response_format=SummaryContent,
    max_tokens=1500,
)

content_v2 = response_v2.choices[0].message.parsed

summary_v2 = DocumentSummary(
    Author       = content_v2.Author,
    Title        = content_v2.Title,
    Relevance    = content_v2.Relevance,
    Summary      = content_v2.Summary,
    Tone         = content_v2.Tone,
    InputTokens  = response_v2.usage.prompt_tokens,
    OutputTokens = response_v2.usage.completion_tokens,
)

print(f"Enhanced summary generated")
print(f"  Input tokens  : {summary_v2.InputTokens:,}")
print(f"  Output tokens : {summary_v2.OutputTokens:,}")
print()
print("--- Enhanced Summary ---")
print(summary_v2.Summary)

Enhanced summary generated
  Input tokens  : 11,715
  Output tokens : 474

--- Enhanced Summary ---
In light of substantial investments ranging from $30 to $40 billion channelled into generative AI (GenAI) technologies, a consequential phenomenon has emerged, termed the 'GenAI Divide'. This divide highlights a significant disparity within performance metrics attributed to two primary stakeholder categories: buyers, comprising enterprises and mid-market players, and builders, which include startups and consultancies. Alarmingly, a staggering 95% of organizations report negligible returns on their investments, with only 5% of integrated GenAI initiatives facilitating substantial financial advantages. This divergence is not a manifestation of deficiencies in model sophistication or regulatory frameworks, but is fundamentally rooted in the varying strategies employed for implementation across different organizational contexts. While tools such as ChatGPT exhibit extensive adoption—over 80%

In [7]:
print("Running enhanced evaluation...")
eval_v2 = evaluate_summary(document_text, summary_v2.Summary)
print("Evaluation complete")

Output()

Running enhanced evaluation...


Output()

Output()

Output()

Evaluation complete


## 6. Results Comparison

In [8]:
import pandas as pd

metrics   = ["Summarization", "Coherence", "Tonality", "Safety"]
scores_v1 = [eval_v1.SummarizationScore, eval_v1.CoherenceScore,
             eval_v1.TonalityScore,       eval_v1.SafetyScore]
scores_v2 = [eval_v2.SummarizationScore, eval_v2.CoherenceScore,
             eval_v2.TonalityScore,       eval_v2.SafetyScore]

comparison = pd.DataFrame({
    "Metric"   : metrics,
    "v1 Score" : [f"{s:.3f}" for s in scores_v1],
    "v2 Score" : [f"{s:.3f}" for s in scores_v2],
    "Delta"    : [f"{v2-v1:+.3f}" for v1, v2 in zip(scores_v1, scores_v2)],
}).set_index("Metric")

print("=" * 44)
print("  Score Comparison: Initial vs Enhanced")
print("=" * 44)
print(comparison.to_string())
print()
print(f"  Average v1: {sum(scores_v1)/len(scores_v1):.3f}")
print(f"  Average v2: {sum(scores_v2)/len(scores_v2):.3f}")
print(f"  Avg delta : {sum(scores_v2)/len(scores_v2) - sum(scores_v1)/len(scores_v1):+.3f}")
print()
print("--- v2 Evaluation Reasons ---")
for label, reason in [
    ("Summarization", eval_v2.SummarizationReason),
    ("Coherence",     eval_v2.CoherenceReason),
    ("Tonality",      eval_v2.TonalityReason),
    ("Safety",        eval_v2.SafetyReason),
]:
    print(f"[{label}]")
    print(f"  {reason}")
    print()

  Score Comparison: Initial vs Enhanced
              v1 Score v2 Score   Delta
Metric                                 
Summarization    0.500    0.529  +0.029
Coherence        0.718    0.792  +0.074
Tonality         0.343    0.361  +0.019
Safety           0.627    0.722  +0.096

  Average v1: 0.547
  Average v2: 0.601
  Avg delta : +0.054

--- v2 Evaluation Reasons ---
[Summarization]
  The score is 0.53 because the summary includes several pieces of extra information that were not present in the original text, which may lead to misinterpretation or confusion. Additionally, it fails to address key questions regarding the report's recommendations and workforce considerations, indicating a lack of completeness in capturing the original text's intent.

[Coherence]
  The response presents ideas in a logical order, starting with the introduction of the 'GenAI Divide' and moving through various statistics and analyses. Transitions between topics are generally smooth, although some sections 

## 7. Analysis and Reflections

### What worked

- **Structured output** with `beta.chat.completions.parse` reliably returns
  typed Pydantic objects; token counts are directly available from
  `response.usage`, keeping the schema clean.

- **Bureaucratese** is an ideal test tone: its features (passive voice,
  nominalisation, jargon) are distinct enough that the Tonality GEval metric
  can assess them with high confidence.

- **Self-correction via feedback injection** guides the model toward identified
  weaknesses. Passing the score *and* the reason string gives the model
  actionable, specific instructions rather than a vague directive.

### Limitations

- **Evaluator hallucination:** `SummarizationMetric` calls an LLM to judge
  assessment questions. If the evaluator misreads bureaucratic phrasing, scores
  may be noisy.

- **Tone vs. coverage trade-off:** Heavy Bureaucratese can bury key facts,
  potentially reducing the Summarization score even when tone is excellent.

- **Single enhancement round:** One self-correction pass provides uplift but
  is unlikely to reach ceiling scores. Iterative correction would be more robust.

- **Are these controls enough for production?** No. The current setup checks
  style and surface safety but does not address faithfulness (hallucination),
  stakeholder-specific bias, or downstream decision risk. Faithfulness metrics
  and human-in-the-loop review would be required for a production pipeline.